# 04 · Goals & Outcome Model
Phase 4: Dixon-Coles model + XGBoost classifier + ensemble.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from src.data_loader import load_all
from src.dixon_coles import DixonColes, tune_xi
from src.models import XGBOutcomeModel, EnsemblePredictor
np.random.seed(42)

data = load_all(verbose=False)
results_full = data['results_full']
results = data['results_primary']
wc_2022 = results_full[results_full['is_wc'] & (results_full['date'].dt.year == 2022)].copy()
train_dc = results_full[(results_full['date'].dt.year >= 2015) & 
                         (results_full['date'].dt.year < 2022)].copy()
print(f"Dixon-Coles train: {len(train_dc):,} | 2022 WC validation: {len(wc_2022):,}")


## 4.1 Tune Dixon-Coles xi (time-decay)

In [ ]:
print("Grid searching best xi (time-decay parameter)...")
xi_results = tune_xi(train_dc, wc_2022, xi_values=[0.001, 0.003, 0.005])
best_xi = xi_results['best_xi']
print(f"\nBest xi = {best_xi:.3f}  |  Best RPS = {xi_results['best_rps']:.4f}")

plt.figure(figsize=(7, 4))
xi_df = pd.DataFrame(xi_results['results'])
plt.plot(xi_df['xi'], xi_df['rps'], 'o-', color='steelblue', lw=2, ms=7)
plt.axvline(best_xi, color='red', linestyle='--', label=f'Best xi={best_xi}')
plt.xlabel('xi (time-decay)'); plt.ylabel('RPS (lower=better)')
plt.title('RPS vs Time-Decay Parameter'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/plots/xi_tuning.png', dpi=150, bbox_inches='tight')
plt.show()


## 4.2 Fit Final Dixon-Coles Model (All Data 2000-2026)

In [ ]:
train_final = results_full[results_full['date'].dt.year >= 2015].copy()
print(f"Fitting final Dixon-Coles on {len(train_final):,} matches...")

dc_model = DixonColes(xi=best_xi)
dc_model.fit(train_final)
dc_model.save_params()

print(f"home_advantage = {dc_model._home_adv:.4f}")
print(f"rho (low-score correction) = {dc_model._rho:.4f}")
print("\nValidating on 2022 WC...")
val_metrics = dc_model.validate(wc_2022)
print(f"  RPS = {val_metrics['rps']:.4f}  |  Accuracy = {val_metrics['accuracy']:.2%}  |  n = {val_metrics['n']}")

# Show team attack/defense ratings
import json
params = dc_model.params_
team_params = pd.DataFrame([
    {'team': t, 'alpha': v['alpha'], 'beta': v['beta']}
    for t, v in params['teams'].items()
]).sort_values('alpha', ascending=False)
print("\nTop 10 Attack (alpha):")
print(team_params.head(10).to_string(index=False))


## 4.3 Example Predictions from Dixon-Coles

In [ ]:
test_matches = [
    ('France', 'Brazil', True),
    ('Argentina', 'Germany', True),
    ('Spain', 'England', True),
    ('Morocco', 'USA', True),
    ('Japan', 'Ecuador', True),
]
print(f"{'Home':<15} {'Away':<15} {'P(H)':<8} {'P(D)':<8} {'P(A)':<8} {'Score'}")
print("-" * 60)
for home, away, neutral in test_matches:
    try:
        probs = dc_model.predict_outcome_probs(home, away, neutral)
        score = dc_model.predict_scoreline(home, away, neutral)
        print(f"{home:<15} {away:<15} {probs['home']:<8.3f} {probs['draw']:<8.3f} "
              f"{probs['away']:<8.3f} {score[0]}-{score[1]}")
    except Exception as e:
        print(f"{home:<15} {away:<15} ERROR: {e}")


## 4.4 XGBoost Outcome Classifier

In [ ]:
feat_path = '../data/processed/match_features.csv'
feat_df = pd.read_csv(feat_path, parse_dates=['date'])
train_xgb = feat_df[feat_df['date'].dt.year >= 2010].dropna(subset=['outcome'])
print(f"XGBoost training set: {len(train_xgb):,} matches")

xgb_model = XGBOutcomeModel()
print("Running Optuna tuning (50 trials)...")
xgb_model.tune_and_fit(train_xgb, n_trials=50)
xgb_model.save()

print("\nTop feature importances:")
imp = sorted(xgb_model.feature_importance.items(), key=lambda x: -x[1])
for feat, score in imp[:8]:
    print(f"  {feat:<35} {score:.4f}")


## 4.5 Ensemble & SHAP Analysis

In [ ]:
ensemble = EnsemblePredictor(dc_weight=0.6, xgb_weight=0.4)
print("Ensemble: 60% Dixon-Coles + 40% XGBoost")

# SHAP values
try:
    import shap
    shap_vals = xgb_model.get_shap_values(train_xgb.head(500))
    if shap_vals is not None:
        print("\nSHAP summary computed (see plot)")
        # Simple bar plot of mean |SHAP|
        feat_cols_used = [c for c in xgb_model.FEATURE_COLS if c in train_xgb.columns]
        if isinstance(shap_vals, list):
            mean_shap = np.mean([np.abs(sv).mean(0) for sv in shap_vals], axis=0)
        else:
            mean_shap = np.abs(shap_vals).mean(0)
        shap_df = pd.DataFrame({'feature': feat_cols_used, 'importance': mean_shap}).sort_values('importance', ascending=False)
        plt.figure(figsize=(8, 5))
        plt.barh(shap_df['feature'].head(10)[::-1], shap_df['importance'].head(10)[::-1], color='steelblue')
        plt.xlabel('Mean |SHAP value|'); plt.title('Top 10 Features by SHAP Importance')
        plt.tight_layout()
        plt.savefig('../outputs/plots/shap_importance.png', dpi=150, bbox_inches='tight')
        plt.show()
except Exception as e:
    print(f"SHAP skipped: {e}")


## ✅ Phase 4 Complete
- Dixon-Coles model fitted and validated
- XGBoost classifier tuned with Optuna
- Model saved to `outputs/model_artifacts/`